In [ ]:
import pandas as pd

import matplotlib.pyplot as plt

from tokenizers import ByteLevelBPETokenizer

from transformers import PreTrainedTokenizerFast

In [ ]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

df

In [ ]:
# Calcola la distribuzione delle classi nella colonna 'label'
label_counts = df['label'].value_counts()

# Calcola la percentuale di ciascuna classe
total = len(df)
percentages = (label_counts / total) * 100

# Stampa i risultati
print(f"Classes probabilites:\n{percentages}")

### Dataset size basing on the features group

In [ ]:
# Seleziona le colonne di features su cui vuoi calcolare
features_to_check = ['consult', 'medication_D0']

# Calcola quante righe hanno almeno un NaN nelle colonne selezionate
rows_with_nan = df[features_to_check].isna().any(axis=1)

# Conta quante righe hanno almeno un NaN
num_rows_to_remove = rows_with_nan.sum()

# Mostra il numero di righe da rimuovere
print(f'Patients number in the dataset: {df.shape[0]}')
print(f"Number of patients to be deleted: {num_rows_to_remove}")
print(f'Remaining patients: {df.shape[0] - num_rows_to_remove}')

### Unstructured data words distribution

In [ ]:
tokenizer_vocab_path  = r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\pretrained\DutchEHRTokenizer\vocab.json'
tokenizer_merges_path = r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\pretrained\DutchEHRTokenizer\merges.txt'

tokenizer = ByteLevelBPETokenizer(
    vocab = tokenizer_vocab_path,
    merges = tokenizer_merges_path,
    add_prefix_space = True
)

tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer._tokenizer)
tokenizer.add_special_tokens({'pad_token': '<pad>'})

def tokens_count(x):
    
    encoding = tokenizer(
        x, 
        truncation = False,  
    )
    
    return len(encoding['input_ids'])

In [ ]:
df_consults = df['consult'].dropna()

consults_words = df_consults.apply(lambda x: tokens_count(x))

In [ ]:
# Plotta l'istogramma che indica la frequenza del numero di token nei consulti
plt.figure(figsize=(8, 6))
plt.hist(consults_words, bins=20, edgecolor='black')
plt.title('Consults')
plt.xlabel('Tokens number')
plt.ylabel('Frequency')
plt.savefig(r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\assets\images\David_Thesis\consults_freq.pdf')
plt.show()

perc = .90
print(f'Max number of words: {consults_words.max()}')
print(f'Number of tokens to cover the {perc*100}% of the patients consults: {int(consults_words.quantile(perc))}')

---

# Dataset Analysis

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

eligible_samples = df[df['label'] == 1].shape[0]

perc = (eligible_samples / df.shape[0]) * 100

print(f"Number of samples in the dataset: {df.shape[0]}")
print(f"Eligible patients: {perc:.2f}%")
print(f"Not-eligible patients: {100 - perc:.2f}%")

In [ ]:
df_male = df[df['gender'] == 0]

df_female = df[df['gender'] == 1]

perc = (df_male.shape[0] / df.shape[0]) * 100

print(f"Male samples: {perc:.2f}%")
print(f"Female samples: {100 - perc:.2f}%")

In [ ]:
df_medicatie = pd.read_parquet(r'L:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_medicatie_ATC.parquet')

df_medicatie

## Male samples

In [ ]:
age_mean = df_male['age'].mean()
age_std = df_male['age'].std()

print(f"Average Age (std): {age_mean:.2f} ({age_std:.2f})")

## Female samples

In [ ]:
age_mean = df_female['age'].mean()
age_std = df_female['age'].std()

print(f"Average Age (std): {age_mean:.2f} ({age_std:.2f})")